# 03 - Quantum Phase Estimation

**Question:** a gate turns a special state by a hidden fraction $\theta$ of a full turn. Can we find $\theta$, and how much noise can the method take?

$$U|\psi\rangle = e^{2\pi i\theta}|\psi\rangle$$

Here $U$ is a phase gate and its eigenstate $|\psi\rangle$ is $|1\rangle$. We use $\theta = k/2^m$ with $m$ counting qubits, so a noise-free run reads $k$ exactly. At 10 qubits: 9 counting + 1 eigenstate, $\theta = k/512$.

**Score:** share of shots that read the correct bits. **Baseline:** $1/2^m$. **Pass:** score at least 50% of the ideal score.

**Bit order:** the rightmost character is qubit 0.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import qasm2
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit.visualization import plot_histogram

from noisy_qubits.algorithms import qft, qpe
from noisy_qubits.noise import DEVICES, NOISE_TYPES, build_noise_model, slider_label

## 1. The circuit

1. X puts the eigenstate qubit in 1. H puts every counting qubit in an equal mix.
2. Counting qubit j applies the hidden gate 2^j times (1, 2, 4, ...). The phase appears on the **counting** qubit, not on the target. This is called **phase kickback**.
3. The inverse QFT turns the phases into bits.
4. Measure the counting qubits.

Example: 3 counting qubits, k = 3, so theta = 3/8.

In [ ]:
qpe.build(4, 3).draw("mpl", fold=40)

**A shortcut:** applying a phase gate 2^j times is the same as one gate with 2^j times the angle, so each counting qubit needs only one CP gate. This works because we built the hidden gate. For an unknown gate that cannot be combined this way, the circuit would be much longer.

## 2. Why the inverse QFT works

After step 2 the counting qubits are in exactly the state that the QFT makes from the number k. The check below shows this: fidelity 1.0 means the two states are the same. So the inverse QFT gives back k.

In [ ]:
m, k = 3, 3
before_iqft = np.exp(2j * np.pi * k * np.arange(2**m) / 2**m) / np.sqrt(2**m)
from_qft = Statevector.from_label(format(k, f"0{m}b")).evolve(qft.qft_circuit(m))

part = qpe.build(m + 1, k, measure=False)
steps_1_2 = part.copy_empty_like()
barriers = 0
for inst in part.data:
    if inst.operation.name == "barrier":
        barriers += 1
        if barriers == 2:
            break
    steps_1_2.append(inst)
# The eigenstate qubit (top bit) stays 1, so its half of the vector holds the state.
counting = Statevector.from_instruction(steps_1_2).data[2**m:]

print("QPE state vs formula:", round(state_fidelity(Statevector(counting), Statevector(before_iqft)), 6))
print("QPE state vs QFT|k>: ", round(state_fidelity(Statevector(counting), from_qft), 6))

## 3. No noise: matches theory

In [ ]:
for n in range(2, 11):
    r = qpe.run(n, 256, seed=1)
    print(f"n={n:2d} k={r.meta['phase_k']:3d} phase={r.meta['phase']:.6f} "
          f"estimated={r.meta['estimated_phase']:.6f} score={r.score:.3f}")
    assert r.score == 1.0

In [ ]:
r = qpe.run(10, 1024, phase_k=181, seed=1)
plot_histogram(r.counts, title="QPE, k=181, no noise")

## 4. Adding gate noise

In [ ]:
for s in (1, 10, 20, 50, 100):
    r = qpe.run(10, 1024, build_noise_model("uniform", s), seed=11)
    verdict = "PASS" if r.score >= 0.5 * r.ideal_score else "below the pass line"
    print(f"{slider_label('uniform', s):32s} score={r.score:.3f}  {verdict}  estimated phase={r.meta['estimated_phase']:.4f}")

The **estimated phase** is the most frequent reading. It can still be correct when the score is below the pass line, because wrong readings are spread over many outcomes.

## 5. QPE against the QFT round trip

In [ ]:
sliders = [1] + list(range(5, 101, 5))
curves = {name: [mod.run(10, 1024, build_noise_model("uniform", s), seed=11).score for s in sliders]
          for name, mod in (("QPE", qpe), ("QFT round trip", qft))}
plt.figure(figsize=(7, 4))
for name, values in curves.items():
    plt.plot(sliders, values, marker="o", label=name)
plt.axhline(0.5, ls=":", c="red", label="pass line")
plt.axhline(1 / 512, ls="--", c="grey", label="QPE classical baseline (1/512)")
plt.xlabel("noise slider"); plt.ylabel("success rate"); plt.legend(); plt.show()

for name, mod in (("QPE", qpe), ("QFT round trip", qft)):
    ops = mod.build().count_ops()
    print(f"{name:15s} cp={ops.get('cp', 0)} swap={ops.get('swap', 0)} h={ops.get('h', 0)}")

QPE survives more noise here. It runs **one** inverse QFT, while the round trip runs a QFT and an inverse QFT, so it has about half the two-qubit gates. This depends on the one-gate shortcut in section 1.

## 6. Which noise types hurt?

In [ ]:
for t in NOISE_TYPES:
    r = qpe.run(10, 1024, build_noise_model("uniform", 20, t), seed=11)
    print(f"{t:18s} {r.score:.3f}")

## 7. Fake IBM devices (slider 50 = the real device)

In [ ]:
for d in DEVICES:
    r = qpe.run(10, 1024, build_noise_model("device", 50, device=d), device=d, seed=11)
    print(f"{d:14s} {r.score:.3f}")

## 8. QASM export

In [ ]:
print(qasm2.dumps(qpe.build(4, 3)))